# PriceOptima EDA

This notebook is exploratory only. It is not a pipeline stage.
Cleaning, the chronological split, and model fitting live in `src/` and `dvc.yaml`.

Question: what does monthly quantity (`qty`) move with, and which columns would leak the answer?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

raw_path = Path("../data/raw/retail_price.csv")
if not raw_path.exists():
    raw_path = Path("data/raw/retail_price.csv")
df = pd.read_csv(raw_path)
df["month_year"] = pd.to_datetime(df["month_year"], format="%d-%m-%Y")
df = df.sort_values(["month_year", "product_id"]).reset_index(drop=True)
df.shape


## Schema

676 rows, 30 columns, no missing values in the Kaggle file. `month_year` is the first of the month written as day-month-year, so the format string is `%d-%m-%Y`. `product_name_lenght` is spelled that way in the file and the pipeline keeps the spelling.


In [ ]:
df.dtypes


In [ ]:
missing = df.isna().sum()
missing[missing > 0]


## Demand over time

Early 2017 has very few products. A random split would treat those thin months as exchangeable with 2018. The pipeline instead holds out the last 20% of distinct months.


In [ ]:
monthly = df.groupby("month_year", as_index=False).agg(qty=("qty", "sum"), products=("product_id", "nunique"))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(monthly["month_year"], monthly["qty"], marker="o")
ax.set_title("Total quantity by month")
ax.set_ylabel("qty")
fig.autofmt_xdate()
fig.tight_layout()


## Price and quantity

Unit price and quantity are weakly negative, which is the elasticity direction, but the relationship is noisy and differs by category. Competitor prices sit in `comp_1`..`comp_3`.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for category, part in df.groupby("product_category_name"):
    ax.scatter(part["unit_price"], part["qty"], s=16, alpha=0.7, label=category)
ax.set_xlabel("unit_price")
ax.set_ylabel("qty")
ax.set_title("Quantity against unit price")
ax.legend(fontsize=7, loc="upper right")
fig.tight_layout()


## Columns that would leak `qty`

`total_price` correlates with `qty` at about 0.75 because it is revenue. `customers` is the buyer count from the same product-month. Both are excluded from the model matrix in `params.yaml`. `s` repeats for the same product and calendar month a year later, so it is kept as a seasonal index. `volume` does not vary inside a product.


In [ ]:
corr = df[["qty", "total_price", "customers", "unit_price", "lag_price", "s", "volume"]].corr()
corr.round(3)


In [ ]:
print("volume unique per product:", int((df.groupby("product_id")["volume"].nunique() == 1).sum()), "/", df["product_id"].nunique())
print("s unique per product-month:", int(df.groupby(["product_id", df["month_year"].dt.month])["s"].nunique().max()))
print("categories:", df["product_category_name"].nunique(), "products:", df["product_id"].nunique())


## What the pipeline does next

Preprocess parses the date, holds out the last 20% of months, and clips Tukey outliers using train-only fences. Features adds `price_vs_comp`, `price_change`, and log price, and fits the imputer, scaler, and one-hot encoder on train only. Nothing in this notebook is fit on the full frame for modelling.
